## 1. Tool(도구)의 정의와 역할

### 1.1 ReAct에서 Tool이란 무엇인가

ReAct 프레임워크에서 **Tool(도구)** 은 LLM이 외부 세계와 상호작용하기 위해 호출하는 실행 가능한 함수 또는 서비스를 의미한다. LLM은 자연어를 이해하고 생성하는 능력이 뛰어나지만, 다음과 같은 한계를 가진다.

- **수학 연산**: 복잡한 계산에서 오류를 범할 수 있다
- **최신 정보**: 학습 데이터 이후의 정보를 알 수 없다
- **외부 시스템 접근**: 데이터베이스, 파일 시스템, 웹 검색 등에 직접 접근할 수 없다
- **정확한 데이터 조회**: 정확한 수치나 사실 정보를 보장할 수 없다

Tool은 이러한 한계를 보완하여, LLM의 추론 능력과 외부 도구의 실행 능력을 결합하는 핵심 요소이다.

### 1.2 Tool의 필수 요소

모든 Tool은 다음 5가지 요소를 갖추어야 한다.

| 요소 | 설명 | 예시 |
|------|------|------|
| **이름(Name)** | 도구를 식별하는 고유한 이름 | `Calculator`, `WebSearch` |
| **설명(Description)** | 도구의 기능을 LLM이 이해할 수 있도록 기술 | "수학 수식을 계산합니다" |
| **입력 스키마(Input Schema)** | 도구가 받는 입력의 형식과 제약 조건 | `{"type": "string"}` |
| **실행 로직(Execute Logic)** | 입력을 받아 실제 작업을 수행하는 코드 | `eval(expression)` |
| **출력 형식(Output Format)** | 결과를 문자열로 반환하는 규칙 | `"계산 결과: 42"` |

### 1.3 Tool이 LLM의 한계를 보완하는 방식

ReAct에서 LLM과 Tool의 협업은 다음과 같은 흐름을 따른다.

```
LLM(Thought) -> 어떤 도구를 사용할지 결정
LLM(Action)  -> 도구 이름과 입력값을 지정
Tool(실행)    -> 실제 연산/조회 수행
LLM(관찰)    -> 도구 결과를 받아 다음 추론에 활용
```

이 구조에서 LLM은 **"무엇을 해야 하는가"**를 결정하는 두뇌 역할을 하고, Tool은 **"실제로 수행"**하는 손과 발 역할을 한다.

## 2. Tool 스키마 설계

### 2.1 JSON Schema 기반 Tool 정의

도구를 체계적으로 정의하기 위해 JSON Schema 형식을 사용한다. 이는 OpenAI의 Function Calling이나 LangChain 등 주요 프레임워크에서도 채택하고 있는 표준 방식이다.

```json
{
  "name": "Calculator",
  "description": "수학 수식을 계산합니다.",
  "parameters": {
    "type": "string",
    "description": "계산할 수학 수식 (예: 2 + 3 * 4)"
  }
}
```

### 2.2 매개변수 타입과 필수/선택 지정

복잡한 도구의 경우, 매개변수를 세부적으로 정의할 수 있다.

```json
{
  "name": "WebSearch",
  "description": "웹에서 정보를 검색합니다.",
  "parameters": {
    "type": "object",
    "properties": {
      "query": {"type": "string", "description": "검색 질의"},
      "max_results": {"type": "integer", "description": "최대 결과 수", "default": 5}
    },
    "required": ["query"]
  }
}
```

### 2.3 Tool 설명문 작성 가이드라인

도구의 설명문은 LLM이 적절한 도구를 선택하는 데 결정적인 역할을 한다. 다음 원칙을 따르는 것이 좋다.

1. **명확성**: 도구가 무엇을 하는지 한 문장으로 설명한다
2. **입력 예시**: 어떤 형식의 입력을 기대하는지 예시를 제공한다
3. **제한 사항**: 도구가 처리할 수 없는 경우를 명시한다
4. **사용 시점**: 언제 이 도구를 사용해야 하는지 안내한다

## 1: 기본 Tool 클래스 설계

이제 파이썬의 추상 클래스(ABC)를 활용하여 Tool의 기본 구조를 설계하고, 세 가지 구체적인 도구를 구현한다.

- **CalculatorTool**: 수학 수식을 평가하는 계산기
- **StringProcessorTool**: 문자열 통계를 분석하는 도구
- **DictionaryTool**: 미리 정의된 사전에서 용어를 검색하는 도구

In [9]:
# abc  (Abstract Base Class)  - 추상클래스
from abc import ABC, abstractmethod
from typing import Any, Dict,Optional
import json
class BaseTool(ABC):  # 추상클래스
    '''React 에이전트에서 사용하는 도구의 기본 클래스'''
    def __init__(self,name:str, description:str):
        self.name = name; self.description = description
    @abstractmethod
    def execute(self, input_text:str) ->str:
        '''도구를 실행하고 결과를 문자열로 반환'''
        pass

    def get_schema(self)->Dict:
        '''도구의 스키마를 반환'''
        return{
            'name' : self.name,
            'description':self.description,
            'parameters':self._get_parameters()
        }
    def _get_parameters(self)->str:
        return {'type':'string','description':'입력테스트'}
    def __repr__(self):  # 객체를 출력할때
        return f'Tool(name={self.name})'   


In [2]:
class Person:
    def __init__(self,name):
        self.name = name
p = Person("hong gil dong")        
print(p)

In [4]:
class Person:
    def __init__(self,name):
        self.name = name
    def __repr__(self):
        return f'ok'
p = Person("hong gil dong")        
print(p)

ok


In [ ]:
class ClaculatorTool(BaseTool):
    def __init__(self):
        super().__init__(
            name='Calcualtor',
            description='수학 수식을 계산합니다. 사칙연산 거듭제곱 나머지연산 등을 지원합니다.'
        )
    def execute(self, input_text:str) ->str:
        try:
            # 안전한 수식 평가를 위해 허용된 연산자만 사용
            allowed_chars = set('0123456789+-*/.() ')
            if not all(c in allowed_chars for c in input_text):
                return f"오류: 허용되지 않는 문자가 포함되어 있습니다."
            result = eval(input_text)
            return f"계산 결과: {result}"
        except Exception as e:
            return f"계산 오류: {str(e)}"
class StringProcessorTool(BaseTool):
    """문자열 처리 도구"""
    
    def __init__(self):
        super().__init__(
            name="StringProcessor",
            description="문자열의 길이, 단어 수, 글자 수 등을 분석합니다."
        )
    
    def execute(self, input_text: str) -> str:
        char_count = len(input_text)
        word_count = len(input_text.split())
        line_count = len(input_text.splitlines())
        return (f"문자 수: {char_count}, 단어 수: {word_count}, "
                f"줄 수: {line_count}")

class DictionaryTool(BaseTool):
    """미리 정의된 사전에서 정보를 검색하는 도구"""
    
    def __init__(self):
        super().__init__(
            name="Dictionary",
            description="내장 사전에서 용어의 정의를 검색합니다."
        )
        self.entries = {
            "인공지능": "인간의 학습, 추론, 지각 능력을 컴퓨터로 구현하는 기술",
            "머신러닝": "데이터로부터 패턴을 학습하여 예측이나 결정을 수행하는 AI의 하위 분야",
            "딥러닝": "다층 신경망을 사용하여 복잡한 패턴을 학습하는 머신러닝의 하위 분야",
            "자연어처리": "컴퓨터가 인간의 언어를 이해하고 생성하는 AI 기술",
            "강화학습": "환경과의 상호작용을 통해 보상을 최대화하는 행동을 학습하는 방법",
            "트랜스포머": "어텐션 메커니즘 기반의 신경망 아키텍처로, 현대 NLP의 기반",
            "LLM": "대규모 텍스트 데이터로 사전 학습된 거대 언어 모델",
            "프롬프트": "LLM에 입력되는 지시문 또는 질의문",
            "ReAct": "추론(Reasoning)과 행동(Acting)을 결합한 프롬프팅 기법"
        }
    
    def execute(self, input_text: str) -> str:
        term = input_text.strip()
        if term in self.entries:
            return f"{term}: {self.entries[term]}"
        # Partial match
        matches = [k for k in self.entries if term in k or k in term]
        if matches:
            results = [f"{k}: {self.entries[k]}" for k in matches]
            return "관련 항목:\n" + "\n".join(results)
        return f"'{term}'에 대한 정보를 찾을 수 없습니다."        



TypeError: Can't instantiate abstract class ClaculatorTool with abstract method execute